# Triangle RIG experiment: simulate and save measurements

This notebook runs the simulation experiment from the original
`3_cores_in_random_intersection_graphs(1).ipynb`. The graph classes and
construction/pruning algorithms live in the first two notebooks; this
notebook adds explicit experiment parameters, independent saved seeds,
parallel execution, and raw CSV measurements.

Use the project's `.venv` kernel. **Run All reruns the experiment and
replaces its CSV file.** Edit the **Experiment settings** cell below
before running. Output paths are relative to the project root. The
following notebook, `04_triangle_analysis.ipynb`, redraws saved results
without rerunning any simulations.

## Model and measured quantity

For each individual, sample $D_i \sim \operatorname{Poisson}(\lambda)$.
Resample the **entire vector** until $\sum_i D_i$ is divisible by three,
then uniformly match its membership half-edges into communities of three
slots. Distinct individuals sharing a community are connected; loops are
omitted and repeated edges are merged. Thus $\lambda$ controls mean
**community memberships**, while the typical graph degree is approximately
$2\lambda$, with finite-size corrections from repeated memberships/edges.

For every $(n,\lambda)$, independently sample several graphs and measure
$$X_{n,\lambda} = \frac{|\operatorname{Core}_3(G_{n,\lambda})|}{n}.$$
The denominator includes isolated individuals. The pruning algorithm
returns a separate graph, leaving the sampled realization intact.

In [ ]:
from pathlib import Path

# Locate sibling notebooks whether the kernel starts here or at the project root.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

# These dependency notebooks define classes/functions without running experiments.
%run "{NOTEBOOK_DIR}/02_algorithms.ipynb"

In [ ]:
import csv
from time import perf_counter

import numpy as np
from joblib import Parallel, delayed

# Keep results in the project root, independent of the kernel's working directory.
PROJECT_ROOT = NOTEBOOK_DIR.parent

## Experiment settings

Edit this cell to choose the experiment. `n` counts all individuals,
including isolates. The lambda grid is equally spaced and includes both
endpoints; lambda is the mean number of community memberships, not the
mean graph degree. `repeats` controls realizations at each lambda.

`seed` initializes independent child streams, while `jobs=1` runs serially
and `jobs=-1` uses all available CPUs. Choose a different `data_file` to
retain earlier measurements. Its path is relative to the project root;
select the same CSV in the analysis notebook's settings cell.

In [ ]:
config = {
    "n": 1000,  # Individuals per graph, including isolates.
    "repeats": 20,  # Independent graph realizations at each lambda.
    "lambda_min": 0.7,  # Mean community memberships at the grid's lower end.
    "lambda_max": 2.5,
    "lambda_count": 41,  # Equally spaced values, including both endpoints.
    "seed": 2026,  # Master seed; every realization saves its own child seed.
    "jobs": 1,  # 1 runs serially; -1 uses all available CPUs.
    "data_file": "results/runs/triangle_transition.csv",  # Replaced on each run.
}
config

## One realization and reproducible repeated trials

`triangle_trial` measures one graph using a supplied seed. The sweep
assigns every trial a child seed **before** sending work to joblib, so
choosing another worker count does not alter the measurements. The row
order is always lambda first, then replicate number (starting at zero).

A saved seed reproduces the sampled structure up to vertex relabeling,
and therefore its core size, in the same code/dependency environment.
`Node` uses random UUIDs as identifiers: those identifiers are not part
of the simulation's reproducibility guarantee. Changing the grid or
repeat count may change the seed assigned to a particular trial.

In [ ]:
CSV_FIELDS = ("model", "n", "lambda", "replicate", "seed", "core_size", "core_fraction")


def triangle_trial(n: int, lam: float, replicate: int, seed: int) -> dict:
    """Sample one triangle RIG and record its 3-core size and replay inputs."""
    if n <= 0:
        raise ValueError("n must be positive")
    rng = np.random.default_rng(seed)
    graph = construct_special_triangle_RIG(n, lam, rng=rng)
    core = construct_k_core(graph, 3)
    core_size = len(core.nodes)
    return {
        "model": "triangle_rig",
        "n": n,
        "lambda": lam,
        "replicate": replicate,
        "seed": seed,
        "core_size": core_size,
        "core_fraction": core_size / n,
    }


def simulate_triangle(
    n: int, lambdas, repeats: int, seed: int = 2026, jobs: int = 1
) -> list[dict]:
    """Return individual measurements in lambda/replicate order.

    ``jobs=1`` runs serially; ``jobs=-1`` uses all available CPUs.
    Seed assignment is independent of joblib's execution order.
    """
    grid = [float(lam) for lam in lambdas]
    if n <= 0 or repeats <= 0 or seed < 0 or jobs == 0:
        raise ValueError(
            "Require positive n/repeats, nonnegative seed, and nonzero jobs"
        )
    if (
        not grid
        or len(set(grid)) != len(grid)
        or any(not np.isfinite(lam) or lam < 0 for lam in grid)
    ):
        raise ValueError("Use distinct, finite, nonnegative lambda values")

    # Give each realization its own independent random stream. Converting
    # to Python int keeps all seed bits when writing/reading the CSV.
    child_seeds = iter(np.random.SeedSequence(seed).spawn(len(grid) * repeats))
    tasks = []
    for lam in grid:
        for replicate in range(repeats):
            trial_seed = int(next(child_seeds).generate_state(1, dtype=np.uint64)[0])
            tasks.append((n, lam, replicate, trial_seed))

    return Parallel(n_jobs=jobs)(delayed(triangle_trial)(*task) for task in tasks)


def save_results(rows: list[dict], path: str | Path) -> None:
    """Replace a CSV with raw trial measurements, creating parent folders."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

## Run the sweep and save every realization

This is the computationally expensive cell. Keeping the raw observations
allows later changes to plots or summaries without regenerating graphs.
Each CSV row stores `model`, `n`, `lambda`, `replicate`, `seed`,
`core_size`, and `core_fraction`. The seed is stored as an integer,
without conversion through floating point.

In [ ]:
lambda_axis = np.linspace(
    config["lambda_min"], config["lambda_max"], config["lambda_count"]
)
started = perf_counter()
rows = simulate_triangle(
    n=config["n"],
    lambdas=lambda_axis,
    repeats=config["repeats"],
    seed=config["seed"],
    jobs=config["jobs"],
)
data_path = PROJECT_ROOT / config["data_file"]
save_results(rows, data_path)
print(
    f"Saved {len(rows)} realizations to {data_path} "
    f"in {perf_counter() - started:.1f} seconds."
)

Continue with `04_triangle_analysis.ipynb` to read this CSV and compare the
distribution of core fractions across the lambda grid. To study finite
size effects, repeat with other values of `n` and choose separate output
paths in the configuration so that earlier experiments are retained.